# DAGER Attack Walkthrough

This notebook dissects the DAGER pipeline step by step on a small sentence pair and a longer sentence pair. The goal is to inspect the intermediate objects that matter for reconstruction:

1. tokenization and labels
2. true gradients
3. rank and subspace recovery (`R_Qs`)
4. L1 candidate filtering per position
5. decoder candidate reconstruction

Tip: keep `batch_size` equal to the number of sentences you pass in manually.

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from notebooks.attack_walkthrough_helpers import (
    SMALL_SENTENCES,
    LONG_SENTENCES,
    build_attack_args,
    make_labels,
    trace_dager_attack,
)

pd.set_option('display.max_colwidth', 200)

## Small Sentences

These are intentionally short so we can inspect each position without a huge candidate table.

In [ ]:
small_sentences = SMALL_SENTENCES
small_args = build_attack_args(
    dataset='sst2',
    batch_size=len(small_sentences),
    model_path='gpt2',
    extra_args=['--rank_tol', '1e-8', '--max_ids', '64'],
)
small_labels = make_labels(len(small_sentences), small_args.device, label=0)
small_trace = trace_dager_attack(small_args, small_sentences, labels=small_labels)

print('Rank:', small_trace['rank'])
small_trace['reference_table']

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
small_trace['l1_trace']['candidate_counts']

In [ ]:
small_trace['decoder_candidates'].head(20)

## Long Sentences

These are harder because the number of positions grows and DAGER has more chances to lose the exact candidate path.

In [ ]:
long_sentences = LONG_SENTENCES
long_args = build_attack_args(
    dataset='sst2',
    batch_size=len(long_sentences),
    model_path='gpt2',
    extra_args=['--rank_tol', '1e-8', '--max_ids', '64'],
)
long_labels = make_labels(len(long_sentences), long_args.device, label=0)
long_trace = trace_dager_attack(long_args, long_sentences, labels=long_labels)

print('Rank:', long_trace['rank'])
long_trace['reference_table']

In [ ]:
long_trace['l1_trace']['candidate_counts'].head(40)

In [ ]:
long_trace['decoder_candidates'].head(20)

## What To Look For

- If `rank` is `None`, the attack cannot start.
- If L1 candidate counts drop to `0` early, DAGER has lost the correct path.
- If decoder candidates are mostly short or repetitive, L2/decoder filtering is struggling.
- Compare small vs long cases to see how the candidate path gets more brittle as sentence length grows.